# 01 - Cierre heuristico seguro

Este notebook muestra la baseline local del modulo `realtime`: contratos, cierre de oracion y metricas offline. No usa GPU, Ollama ni APIs externas.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "realtime").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

## Contrato de salida

El cierre siempre devuelve una de tres acciones: `commit`, `wait` o `low_confidence`. Si algo falla, el fallback seguro es `wait`.

In [ ]:
from realtime.src.contracts import CommitAction, PartialHypothesis
from realtime.src.cierre import HeuristicClosureProvider

provider = HeuristicClosureProvider()
examples = [
    "",
    "yo creo que",
    "bueno bueno bueno bueno",
    "ayer fuimos a la cancha y estuvo buenisimo.",
    "vos tenes razon che gracias por avisarme hoy",
]

for text in examples:
    decision = provider.decide(PartialHypothesis(partial_text=text))
    print({"text": text, **decision.to_dict()})

## Evaluacion sintetica

Los casos demo cubren vacio, conector colgante, repeticion, cierre por puntuacion y cierre por contexto suficiente.

In [ ]:
import json
from realtime.src.evaluar_cierre import DEMO_CASES, evaluate_closure

summary = evaluate_closure(DEMO_CASES, provider)
compact = {k: summary[k] for k in [
    "count", "accuracy", "commit_precision", "commit_recall",
    "premature_commit_rate", "unnecessary_wait_rate", "low_confidence_recall",
    "fallbacks", "latency_ms"
]}
print(json.dumps(compact, indent=2, ensure_ascii=False))

In [ ]:
for row in summary["rows"]:
    print(row["case"], "expected=", row["expected"], "predicted=", row["predicted"], "reason=", row["reason"])

## Lectura

- La baseline privilegia no cortar de mas: ante duda devuelve `wait`.
- `low_confidence` queda reservado para texto vacio, muy corto o repetitivo.
- Esta baseline no reemplaza al LLM; sirve como piso seguro y medible para comparar Qwen/Ollama despues.